# 02 — Experimentos, tabelas e figuras (Kaggle, CPU)

Inputs: a competição **e** as saídas de **todos** os shards do notebook 01 (características).
Os shards são juntados automaticamente numa pasta única. Gera `results/` (tabelas CSV,
`RESULTS.md`, figuras e a galeria de erros, que precisa ler os DICOMs da competição).

Tempo medido em 4 CPUs com 3.000 exames: ≈ 3 h (a grade por exame leva ≈ 1,5 h, o nível de corte
e as ablações o resto). Semente fixa (`seed: 42` em `configs/config.yaml`).

In [ ]:
# === Setup (Kaggle) ==========================================================
# Pré-requisitos: aceitar as regras da competição "RSNA STR Pulmonary Embolism Detection"
# e adicioná-la como Input deste notebook (+ Add Input -> Competitions).
import glob, os, sys, subprocess, shutil
REPO_URL = "https://github.com/silvarawenne/ap2-topicos.git"
REPO_BRANCH = "claude/peaceful-curie-mqrljz"   # troque para main após o merge
REPO = "/kaggle/working/ap2-topicos"

if not os.path.exists(REPO):
    # opção sem Internet: o repositório enviado como Dataset (tem configs/config.yaml na raiz)
    hits = glob.glob("/kaggle/input/**/configs/config.yaml", recursive=True)
    if hits:
        shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), REPO)
    else:  # exige "Internet on" nas configurações do notebook
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/requirements-kaggle.txt"], check=True)
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src")
# os dados da competição são achados sozinhos (ou defina PE_DATA_ROOT)
hits = glob.glob("/kaggle/input/**/train.csv", recursive=True)
if hits:
    os.environ.setdefault("PE_DATA_ROOT", os.path.dirname(hits[0]))
print("repo:", REPO, "| dados:", os.environ.get("PE_DATA_ROOT"))


In [ ]:
# Junta as características de todos os shards (Inputs) numa pasta única
merged = "/kaggle/working/features"
os.makedirs(f"{merged}/slices", exist_ok=True)
feats = [p.rsplit("/slices", 1)[0] for p in glob.glob("/kaggle/input/**/slices", recursive=True)]
assert feats, "adicione como Input a saída dos notebooks de extração (01)"
for f in feats:
    for p in glob.glob(f"{f}/slices/*.parquet"):
        dst = f"{merged}/slices/{os.path.basename(p)}"
        if not os.path.exists(dst):
            os.symlink(p, dst)
    if os.path.exists(f"{f}/exams.csv"):
        shutil.copy(f"{f}/exams.csv", merged)
    if os.path.exists(f"{f}/splits"):      # a partição usada na extração é a do repositório
        shutil.copytree(f"{f}/splits", "splits", dirs_exist_ok=True)
os.environ["PE_FEATURES_DIR"] = merged
print(len(feats), "shards |", len(os.listdir(f"{merged}/slices")), "blocos")

In [ ]:
!python -W ignore scripts/03_run_experiments.py

In [ ]:
!python -W ignore scripts/04_make_figures.py --gallery
from IPython.display import Markdown, Image, display
display(Markdown(open("results/RESULTS.md").read()))
for f in sorted(glob.glob("results/figures/*.png")):
    display(Image(f, width=700))

In [ ]:
!cd /kaggle/working && zip -qr tp1_results.zip ap2-topicos/results ap2-topicos/splits && ls -la tp1_results.zip